# Task 1 — Evaluation Rubric

## Context

We need to evaluate LLM-generated product descriptions (50–90 words each) for e-commerce products. Each product has a name, structured attributes, material, and warranty. Descriptions are rated on 7 criteria using a three-level scale: **good**, **ok**, **bad**.

## 1. Criterion Definitions

Each criterion has explicit boundaries for good / ok / bad so that different evaluators arrive at the same verdict.

| Criterion | Good | Ok | Bad |
|-----------|------|----|-----|
| **Fluency** | Reads naturally; no awkward phrasing | 1–2 clunky phrases but still understandable | Choppy, hard to follow, or incoherent |
| **Grammar** | Zero spelling or punctuation errors | 1–2 minor errors that don't change meaning | 3+ errors, or any error that changes meaning |
| **Tone** | Friendly, persuasive, professional sales voice throughout | Mostly appropriate but uneven in places | Wrong tone — robotic, aggressive, or unprofessional |
| **Length** | 50–90 words | 40–49 or 91–110 words | < 40 or > 110 words |
| **Grounding** | All claims supported by the provided product info; nothing fabricated | Minor reasonable extrapolation not explicitly in the data | Fabricated features or claims not in the input |
| **Latency** | <= 2 s per call | 2–5 s per call | > 5 s per call |
| **Cost** | <= $0.001 per call | $0.001–$0.005 per call | > $0.005 per call |

Latency and Cost thresholds are preliminary estimates for small models (~9B) on Nebius Token Factory. They will be recalibrated after Task 2 once we have real measurements.

## 2. Pass / Fail Definition

### 2a. Go / No-Go Rules

A single criterion can trigger automatic failure regardless of other scores:

| Criterion | Trigger | Rationale |
|-----------|---------|-----------|
| **Grounding** | Rated **bad** | Fabricated product info is a legal and trust liability in e-commerce — it can mislead customers and cause returns. |
| **Length** | Rated **bad** | Length is a hard spec. A description far outside 50–90 words means the model failed a basic instruction. |

### 2b. Cumulative Pass Bar

If no go/no-go rule fires, a description passes when both conditions hold:

1. At least **4 out of 7** criteria rated **good**
2. At most **1** criterion rated **bad**

Otherwise the description fails.

### 2c. Formula

```python
def final_score(ratings: dict[str, str]) -> str:
    # Go / no-go
    if ratings["Grounding"] == "bad" or ratings["Length"] == "bad":
        return "fail"

    # Cumulative pass bar
    count_good = sum(1 for v in ratings.values() if v == "good")
    count_bad  = sum(1 for v in ratings.values() if v == "bad")

    if count_good >= 4 and count_bad <= 1:
        return "pass"
    return "fail"
```

### 2d. Decision Matrix

| Scenario | Result | Reason |
|----------|--------|--------|
| All 7 good | Pass | Exceeds pass bar |
| 5 good, 1 ok, 1 bad (not Grounding/Length) | Pass | 5 >= 4, 1 bad <= 1, no go/no-go |
| 4 good, 2 ok, 1 bad (not Grounding/Length) | Pass | 4 >= 4, 1 bad <= 1, no go/no-go |
| 4 good, 1 ok, Grounding = bad | Fail | Go/no-go triggered |
| 4 good, 1 ok, Length = bad | Fail | Go/no-go triggered |
| 3 good, 4 ok, 0 bad | Fail | 3 good < 4 minimum |
| 4 good, 1 ok, 2 bad (not Grounding/Length) | Fail | 2 bad > 1 maximum |

## 3. Code Reference

The rubric and scoring function are encoded in `shared/constants.py` so all later tasks (manual eval, judge model, analysis) import the same definitions.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from shared.constants import CRITERIA, final_score

test_cases = [
    ("All good",                          {c: "good" for c in CRITERIA},                                                                                "pass"),
    ("5 good, 1 ok, 1 bad (Fluency)",     {"Fluency": "bad", "Grammar": "good", "Tone": "good", "Length": "good", "Grounding": "good", "Latency": "good", "Cost": "ok"},  "pass"),
    ("4 good, 2 ok, 1 bad (Cost)",        {"Fluency": "good", "Grammar": "good", "Tone": "ok", "Length": "good", "Grounding": "good", "Latency": "ok", "Cost": "bad"},    "pass"),
    ("Grounding bad -> auto fail",        {"Fluency": "good", "Grammar": "good", "Tone": "good", "Length": "good", "Grounding": "bad", "Latency": "good", "Cost": "ok"},  "fail"),
    ("Length bad -> auto fail",           {"Fluency": "good", "Grammar": "good", "Tone": "good", "Length": "bad", "Grounding": "good", "Latency": "good", "Cost": "ok"},   "fail"),
    ("3 good, 4 ok -> too few good",      {"Fluency": "good", "Grammar": "good", "Tone": "good", "Length": "ok", "Grounding": "ok", "Latency": "ok", "Cost": "ok"},       "fail"),
    ("4 good, 1 ok, 2 bad -> too many bad", {"Fluency": "bad", "Grammar": "good", "Tone": "good", "Length": "good", "Grounding": "good", "Latency": "bad", "Cost": "ok"}, "fail"),
]

for desc, ratings, expected in test_cases:
    result = final_score(ratings)
    mark = "OK" if result == expected else "FAIL"
    print(f"[{mark}] {desc}: {result}")
